Goal: Implement a small semantic search over FAQs using sentence embeddings (no fine‑tuning), then discuss when this is better than keyword search.

Tasks:

1. Load data and inspect the questions and answers with pandas.

2. Use a sentence‑embedding model (for example from sentence-transformers, such as all-MiniLM-L6-v2) to:

Create embeddings for all FAQ questions.
Store them in an array faq_embeddings.

3. Write a function search_faq(query, top_k=2) that:

Embeds the user query.
Computes cosine similarity between query embedding and all faq_embeddings.
Returns the top top_k matching FAQs (id, question, answer, similarity score).

4.Test the function with at least two queries, such as:

"I forgot my password, what should I do?"
"How can I see monthly revenue numbers?"

5. Concept reflection (in comments/markdown or here):

Why would embeddings + cosine similarity handle rephrased questions better than pure keyword LIKE '%password%' queries?
In a real system, when might you still combine this with traditional filters (e.g., by product, language, or user role)?

No need to use LangChain yet; focus on understanding semantic search with embeddings.

In [9]:
import pandas as pd
from io import StringIO

data = '''id,question,answer
1,"How can I reset my account password?","Go to Settings > Security, click 'Reset Password', and follow the instructions sent to your email."
2,"Where can I download monthly sales reports?","Monthly sales reports are available under Reports > Sales > Monthly in the dashboard."
3,"How do I add a new team member?","Navigate to Admin > Users, click 'Add User', and fill in the required details."
4,"What payment methods are supported?","We currently support credit cards, debit cards, UPI, and net banking."
5,"How do I contact customer support?","You can reach support via the Help Center chat or by emailing support@example.com."'''


df = pd.read_csv(StringIO(data))


In [10]:
df

,id,question,answer
0,1,How can I reset my account password?,"Go to Settings > Security, click 'Reset Passwo..."
1,2,Where can I download monthly sales reports?,Monthly sales reports are available under Repo...
2,3,How do I add a new team member?,"Navigate to Admin > Users, click 'Add User', a..."
3,4,What payment methods are supported?,"We currently support credit cards, debit cards..."
4,5,How do I contact customer support?,You can reach support via the Help Center chat...


In [17]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

faq_embeddings = model.encode(df['question'].to_list(),normalize_embeddings = True)


In [18]:
print(faq_embeddings)

[[-0.00777156 -0.05857887 -0.0558641  ...  0.06587974  0.01528475
  -0.09231293]
 [-0.01750006 -0.06957788 -0.14748512 ...  0.03706531  0.0060654
   0.01561121]
 [-0.05870421 -0.09229027 -0.05012619 ...  0.02956694  0.00451528
  -0.05807064]
 [-0.0215135   0.0033785  -0.02416563 ... -0.07397363  0.15292227
   0.00335583]
 [-0.06169292  0.00640712  0.04887156 ...  0.00743746  0.00226341
  -0.00140932]]


In [28]:
from numpy.linalg import norm
import numpy as np

def search_faq(query, top_k= 2):
    query_emb = model.encode(query,normalize_embeddings = True)
    similarity = np.dot(faq_embeddings, query_emb)

    top_indices = similarity.argsort()[-top_k:][::-1]

    results = []

    for i in top_indices:
        results.append({
            "id":df.iloc[i]['id'],
            "question":df.iloc[i]['question'],
            "answer":df.iloc[i]['answer'],
            "similarity": float(similarity[i])
        })
    return results


In [23]:
print(search_faq("I forgot my password, what should I do?"))

[{'id': 1, 'question': 'How can I reset my account password?', 'answer': "Go to Settings > Security, click 'Reset Password', and follow the instructions sent to your email.", 'similarity': 0.789047122001648}, {'id': 5, 'question': 'How do I contact customer support?', 'answer': 'You can reach support via the Help Center chat or by emailing support@example.com.', 'similarity': 0.40843600034713745}]


In [27]:
print(search_faq("How can I see monthly revenue numbers?"))

[{'id': 2, 'question': 'Where can I download monthly sales reports?', 'answer': 'Monthly sales reports are available under Reports > Sales > Monthly in the dashboard.', 'similarity': 0.642406702041626}, {'id': 5, 'question': 'How do I contact customer support?', 'answer': 'You can reach support via the Help Center chat or by emailing support@example.com.', 'similarity': 0.1577528715133667}]


1. Why would embeddings + cosine similarity handle rephrased questions better than pure keyword LIKE '%password%' queries?

Because embeddings are converted into vector dimensions so that its semantic meaning is understood to the system. and using cosine similarity the most relevant query are matched, which says its very similar to this query. it would be mapped and hence it would make questions understand better than pure keyword based

2. In a real system, when might you still combine this with traditional filters (e.g., by product, language, or user role)?

Embeddings can understand the meaning but they dont kknow about the context(What context like which product the user is using), permissions(what role is the user), preferences(langugae, region etc) without filtrs it might return semantically correct but can sometimes be irrelevant or inappropriate for the user